In [0]:
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
df=spark.createDataFrame(data,columns)


In [0]:
df.write.format("delta")\
    .mode("append")\
        .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/deltalogs") 

In [0]:
bronze_df=spark.read.format("delta").load("/Volumes/workspace/default/deltalogs")
display(bronze_df)

In [0]:
from pyspark.sql.functions import col, to_date, trim, initcap

# Load from Bronze
df_bronze = spark.read.format("delta").load("/Volumes/workspace/default/deltalogs")

# 1. Data Type Fix & NULL Handling
df_silver_clean = df_bronze \
    .withColumn("amount", col("amount").cast("int")) \
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd")) \
    .fillna({"amount": 0, "city": "Unknown"}) \
    .filter(col("amount") >= 0) # Data Validation: Remove negatives

df_silver_clean.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# 2. Remove Duplicates (Keep latest record based on date)
window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

df_silver_final = df_silver_clean \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num") \
    .withColumn("city", initcap(trim(col("city")))) # Standardization

In [0]:
silver_table_path = "/Volumes/workspace/default/deltalogs"

df_silver_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print("Silver Layer: Data cleaned and deduplicated.")
df_silver_final.display()

In [0]:
from pyspark.sql.functions import sum, count, desc

# Total sales by Product
# Total sales by Product
product_sales = df_silver_final.groupBy("product") \
    .agg(sum("amount").alias("total_revenue")) \
    .orderBy(desc("total_revenue"))

# Total sales by Category
category_sales = df_silver_final.groupBy("category") \
    .agg(sum("amount").alias("total_revenue")) \
    .orderBy(desc("total_revenue"))

# Total sales by City
city_sales = df_silver_final.groupBy("city") \
    .agg(sum("amount").alias("total_revenue")) \
    .orderBy(desc("total_revenue"))

city_sales.display()

In [0]:
customer_metrics = df_silver_final.groupBy("customer_id") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("amount").alias("total_spent")
    ) \
    .orderBy(desc("total_spent"))

customer_metrics.display()

In [0]:
# 1. Top Selling Product (Column is 'product', not 'product_name')
# Ensure 'product_sales' was created using 'product' in the cell above
top_product = product_sales.limit(1)

# 2. Top Spending Customer
top_customer = customer_metrics.limit(1)

# Collect results safely
top_prod_row = top_product.collect()[0]
top_cust_row = top_customer.collect()[0]

print(f"Top Product: {top_prod_row['product']}")
print(f"Top Customer ID: {top_cust_row['customer_id']}")
print(f"Highest Spending: {top_cust_row['total_spent']}")

In [0]:
# Saving to Gold Layer
# Ensure we are saving the dataframes created in previous steps
product_sales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/deltalogs")

customer_metrics.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/deltalogs")

print("Gold Layer: Business insights generated and stored successfully.")